In [6]:
%pip install --upgrade pip

  Using cached pip-25.2-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install torch torch_geometric pandas numpy

  Using cached torch_geometric-2.6.1-py3-none-any.whl.metadata (63 kB)
   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
    --------------------------------------- 5.8/241.3 MB 32.0 MB/s eta 0:00:08
   -- ------------------------------------- 15.5/241.3 MB 40.5 MB/s eta 0:00:06
   --- ------------------------------------ 24.1/241.3 MB 39.2 MB/s eta 0:00:06
   ----- ---------------------------------- 32.0/241.3 MB 38.3 MB/s eta 0:00:06
   ------ --------------------------------- 40.6/241.3 MB 38.6 MB/s eta 0:00:06
   ------- -------------------------------- 47.7/241.3 MB 37.5 MB/s eta 0:00:06
   --------- ------------------------------ 57.1/241.3 MB 38.8 MB/s eta 0:00:05
   ---------- ----------------------------- 65.8/241.3 MB 39.2 MB/s eta 0:00:05
   ------------ --------------------------- 72.9/241.3 MB 38.4 MB/s eta 0:00:05
   ------------- -------------------------- 82.1/241.3 MB 38.8 MB/s eta 0:00:05
   -------------- ------------------------- 89.9/241.3 MB 3

In [8]:
import os, numpy as np, pandas as pd, torch
from torch_geometric.datasets import WebKB

def ensure_dir(d): os.makedirs(d, exist_ok=True)
def to_np(x): return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def export_webkb(name: str, root: str, out_dir: str):
    ds = WebKB(root=root, name=name) # "Cornell", "Texas", "Wisconsin"
    data = ds[0]
    out = os.path.join(out_dir, name); ensure_dir(out)
    X = to_np(data.x); pd.DataFrame(X).to_csv(os.path.join(out, "features.csv"), header=False, index=False)
    y = to_np(data.y).reshape(-1); pd.DataFrame(y).to_csv(os.path.join(out, "labels.csv"), header=False, index=False)
    edges = to_np(data.edge_index).T; np.savetxt(os.path.join(out, "edges.txt"), edges, fmt="%d")
    with open(os.path.join(out, "meta.txt"), "w") as f:
        f.write(f"name={name}\n")
        f.write(f"num_nodes={data.num_nodes}\n")
        f.write(f"num_features={X.shape[1]}\n")
        f.write(f"num_classes={int(y.max())+1}\n")
        f.write(f"num_edges={edges.shape[0]}\n")
    print(f"✓ Exported {name}")

export_webkb("Texas", "./pyg_datasets", "./webkb_text_min")

Processing...
Done!


✓ Exported Texas


In [9]:
# Minimal WebKB → text exporter (ONLY edges/features/labels)
import os, numpy as np, torch
from torch_geometric.datasets import WebKB

def ensure_dir(d): os.makedirs(d, exist_ok=True)
def to_np(x): return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def export_webkb_min(name="Texas", root="./pyg_datasets", out_dir="./webkb_text_min"):
    ds = WebKB(root=root, name=name) # "Cornell", "Texas", "Wisconsin"
    data = ds[0] # single graph
    out = os.path.join(out_dir, name); ensure_dir(out)
    
    # 1) edges.txt (0-based, two cols)
    edges = to_np(data.edge_index).T # [E, 2]
    np.savetxt(os.path.join(out, "edges.txt"), edges, fmt="%d")
    
    # 2) features.csv (N x F, no header)
    X = to_np(data.x) # [N, F]
    np.savetxt(os.path.join(out, "features.csv"), X, delimiter=",", fmt="%.10g")
    
    # 3) labels.csv (N rows, one int)
    y = to_np(data.y).reshape(-1) # [N]
    np.savetxt(os.path.join(out, "labels.csv"), y, fmt="%d")
    
    print(f"✓ ONLY wrote edges.txt, features.csv, labels.csv -> {out}")

# Example:
export_webkb_min("Texas")
export_webkb_min("Cornell")
export_webkb_min("Wisconsin")

✓ ONLY wrote edges.txt, features.csv, labels.csv -> ./webkb_text_min\Texas


Processing...
Done!


✓ ONLY wrote edges.txt, features.csv, labels.csv -> ./webkb_text_min\Cornell


Processing...


✓ ONLY wrote edges.txt, features.csv, labels.csv -> ./webkb_text_min\Wisconsin


Done!


In [6]:
# Minimal WikiCS / Actor → text exporters (ONLY edges/features/labels)
import os, numpy as np, torch
from torch_geometric.datasets import WikiCS, Actor

def ensure_dir(d): os.makedirs(d, exist_ok=True)
def to_np(x): return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def export_wikics_min(root="./pyg_datasets", out_dir="./wikics_text_min"):
    ds = WikiCS(root=root) # single graph with official masks (ignored here)
    data = ds[0]
    out = os.path.join(out_dir, "WikiCS"); ensure_dir(out)
    
    # 1) edges.txt (0-based, two cols)
    edges = to_np(data.edge_index).T # [E, 2]
    np.savetxt(os.path.join(out, "edges.txt"), edges, fmt="%d")
    
    # 2) features.csv (N x F, no header)
    X = to_np(data.x) # [N, F]
    np.savetxt(os.path.join(out, "features.csv"), X, delimiter=",", fmt="%.10g")
    
    # 3) labels.csv (N rows, one int)
    y = to_np(data.y).reshape(-1) # [N]
    np.savetxt(os.path.join(out, "labels.csv"), y, fmt="%d")
    
    print(f"✓ ONLY wrote edges.txt, features.csv, labels.csv -> {out}")

In [7]:
def export_actor_min(root="./pyg_datasets", out_dir="./actor_text_min"):
    ds = Actor(root=root) # single graph, no official masks used here
    data = ds[0]
    out = os.path.join(out_dir, "Actor"); ensure_dir(out)
    
    # 1) edges.txt (0-based, two cols)
    edges = to_np(data.edge_index).T
    np.savetxt(os.path.join(out, "edges.txt"), edges, fmt="%d")
    
    # 2) features.csv (N x F, no header)
    X = to_np(data.x)
    np.savetxt(os.path.join(out, "features.csv"), X, delimiter=",", fmt="%.10g")
    
    # 3) labels.csv (N rows, one int)
    y = to_np(data.y).reshape(-1)
    np.savetxt(os.path.join(out, "labels.csv"), y, fmt="%d")
    
    print(f"✓ ONLY wrote edges.txt, features.csv, labels.csv -> {out}")

In [ ]:
def export_wiki_min(root="./pyg_datasets", out_dir="./actor_text_min"):
    ds = Actor(root=root) # single graph, no official masks used here
    data = ds[0]
    out = os.path.join(out_dir, "Actor"); ensure_dir(out)
    
    # 1) edges.txt (0-based, two cols)
    edges = to_np(data.edge_index).T
    np.savetxt(os.path.join(out, "edges.txt"), edges, fmt="%d")
    
    # 2) features.csv (N x F, no header)
    X = to_np(data.x)
    np.savetxt(os.path.join(out, "features.csv"), X, delimiter=",", fmt="%.10g")
    
    # 3) labels.csv (N rows, one int)
    y = to_np(data.y).reshape(-1)
    np.savetxt(os.path.join(out, "labels.csv"), y, fmt="%d")
    
    print(f"✓ ONLY wrote edges.txt, features.csv, labels.csv -> {out}")

In [8]:
export_wikics_min(root="./pyg_datasets", out_dir="./wikics_text_min")
export_actor_min(root="./pyg_datasets", out_dir="./actor_text_min")

C:\Users\user\AppData\Local\anaconda3\Lib\site-packages\torch_geometric\datasets\wikics.py:45: UserWarning: The WikiCS dataset now returns an undirected graph by default. Please explicitly specify 'is_undirected=False' to restore the old behavior.
  warnings.warn(
Processing...
Done!


✓ ONLY wrote edges.txt, features.csv, labels.csv -> ./wikics_text_min\WikiCS


Processing...
Done!


✓ ONLY wrote edges.txt, features.csv, labels.csv -> ./actor_text_min\Actor


In [10]:
# Minimal WebKB → text exporter (ONLY edges/features/labels)
import os, numpy as np, torch
from torch_geometric.datasets import WikipediaNetwork

def ensure_dir(d): os.makedirs(d, exist_ok=True)
def to_np(x): return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def export_wiki_min(name="chameleon", root="./pyg_datasets", out_dir="./wiki_text_min"):
    ds = WikipediaNetwork(root=root, name=name)
    data = ds[0] # single graph
    out = os.path.join(out_dir, name); ensure_dir(out)
    
    # 1) edges.txt (0-based, two cols)
    edges = to_np(data.edge_index).T # [E, 2]
    np.savetxt(os.path.join(out, "edges.txt"), edges, fmt="%d")
    
    # 2) features.csv (N x F, no header)
    X = to_np(data.x) # [N, F]
    np.savetxt(os.path.join(out, "features.csv"), X, delimiter=",", fmt="%.10g")
    
    # 3) labels.csv (N rows, one int)
    y = to_np(data.y).reshape(-1) # [N]
    np.savetxt(os.path.join(out, "labels.csv"), y, fmt="%d")
    
    print(f"✓ ONLY wrote edges.txt, features.csv, labels.csv -> {out}")

# Example:
export_wiki_min("chameleon")
export_wiki_min("squirrel")

✓ ONLY wrote edges.txt, features.csv, labels.csv -> ./wiki_text_min\chameleon


Processing...
Done!


✓ ONLY wrote edges.txt, features.csv, labels.csv -> ./wiki_text_min\squirrel


In [4]:
# Minimal CitationFull → text exporter (ONLY edges/features/labels)
# Datasets supported: cora_ml, citeseer, pubmed, dblp
# Files written per dataset:
# edges.txt (E x 2, 0-based indices)
# features.csv (N x F, float, no header)
# labels.csv (N rows, int, no header)

import os
import numpy as np
import torch
from torch_geometric.datasets import CitationFull

def ensure_dir(d):
    os.makedirs(d, exist_ok=True)

def to_np(x):
    if torch.is_tensor(x):
        x = x.detach().cpu()
        # Handle sparse tensors if any appear
        if x.is_sparse:
            x = x.to_dense()
        return x.numpy()
    return np.asarray(x)

def export_citation_min(name="cora_ml", root="./pyg_datasets", out_dir="./citation_text_min"):

    name = str(name).lower()
    allowed = {"cora", "citeseer", "cora_ml", "dblp", "pubmed"}
    if name not in allowed:
        raise ValueError(f"name must be one of {sorted(allowed)} (got: {name})")
    
    ds = CitationFull(root=root, name=name)
    data = ds[0] # single graph
    
    out = os.path.join(out_dir, name)
    ensure_dir(out)
    
    # 1) edges.txt (0-based, two cols)
    # CitationFull edges are directed; your R code symmetrizes later, so we keep them raw.
    edges = to_np(data.edge_index).T.astype(np.int64) # [E, 2]
    np.savetxt(os.path.join(out, "edges.txt"), edges, fmt="%d")
    
    # 2) features.csv (N x F, no header)
    X = to_np(data.x) # [N, F]
    # Ensure 2D (some loaders could give (N,) — just in case)
    if X.ndim == 1:
        X = X[:, None]
    np.savetxt(os.path.join(out, "features.csv"), X, delimiter=",", fmt="%.10g")
    
    # 3) labels.csv (N rows, one int)
    y = to_np(data.y).reshape(-1).astype(np.int64) # [N]
    np.savetxt(os.path.join(out, "labels.csv"), y, fmt="%d")
    
    print(f"✓ Wrote edges.txt / features.csv / labels.csv to {out}")
    print(f" Shapes: edges={edges.shape}, X={X.shape}, y={y.shape}")

# Examples (uncomment what you need):
# export_citation_min("cora_ml", root="./pyg_datasets", out_dir="./citation_text_min")
# export_citation_min("citeseer", root="./pyg_datasets", out_dir="./citation_text_min")
# export_citation_min("pubmed", root="./pyg_datasets", out_dir="./citation_text_min")
# export_citation_min("dblp", root="./pyg_datasets", out_dir="./citation_text_min")

if __name__ == "__main__":
# Batch export all four (comment out to pick selectively)
    for nm in ["cora_ml", "citeseer", "pubmed", "dblp"]:
        export_citation_min(nm)

Processing...
Done!


✓ Wrote edges.txt / features.csv / labels.csv to ./citation_text_min\cora_ml
 Shapes: edges=(16316, 2), X=(2995, 2879), y=(2995,)


Processing...
Done!


✓ Wrote edges.txt / features.csv / labels.csv to ./citation_text_min\citeseer
 Shapes: edges=(10674, 2), X=(4230, 602), y=(4230,)


Processing...
Done!


✓ Wrote edges.txt / features.csv / labels.csv to ./citation_text_min\pubmed
 Shapes: edges=(88648, 2), X=(19717, 500), y=(19717,)


Processing...
Done!


✓ Wrote edges.txt / features.csv / labels.csv to ./citation_text_min\dblp
 Shapes: edges=(105734, 2), X=(17716, 1639), y=(17716,)


In [5]:
# Minimal Amazon → text exporter (ONLY edges/features/labels)
# Matches your Chameleon/Squirrel exporter format.
# - edges.txt : shape [E, 2], 0-based, space-separated ints
# - features.csv : shape [N, F], comma-separated floats
# - labels.csv : shape [N], one int per line

import os
import numpy as np
import torch
from torch_geometric.datasets import Amazon

def ensure_dir(d: str) -> None:
    os.makedirs(d, exist_ok=True)

def to_np(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def export_amazon_min(name: str = "Computers",
                        root: str = "./pyg_datasets",
                        out_dir: str = "./amazon_min") -> None:

# 1) load single-graph dataset
    ds = Amazon(root=root, name=name)
    data = ds[0] # Amazon is a single heterogeneous graph per split
    
    # 2) prepare output folder (e.g., ./amazon_min/computers)
    out = os.path.join(out_dir, name.lower())
    ensure_dir(out)
    
    # 3) EDGES — 0-based indices, two columns [src, dst]
    edge_index = to_np(data.edge_index).T # [E, 2]
    edge_index = edge_index.astype(np.int64, copy=False)
    if edge_index.ndim != 2 or edge_index.shape[1] != 2:
        raise ValueError(f"edge_index expected shape [E,2], got {edge_index.shape}")
    np.savetxt(os.path.join(out, "edges.txt"), edge_index, fmt="%d")

# 4) FEATURES — dense node features [N, F]
    X = to_np(data.x) # [N, F]
    if X.ndim != 2:
        raise ValueError(f"x expected rank-2 [N,F], got {X.shape}")
    np.savetxt(os.path.join(out, "features.csv"), X, delimiter=",", fmt="%.10g")

# 5) LABELS — integer labels [N]
    y = to_np(data.y).reshape(-1).astype(np.int64, copy=False)
    if y.shape[0] != X.shape[0]:
        raise ValueError(f"labels length {y.shape[0]} != number of nodes {X.shape[0]}")
    np.savetxt(os.path.join(out, "labels.csv"), y, fmt="%d")
    
    # 6) quick summary
    print(f"✓ Wrote edges.txt, features.csv, labels.csv to: {out}")
    print(f" Nodes: {X.shape[0]:,} | Feats: {X.shape[1]:,} | Edges: {edge_index.shape[0]:,} | Classes: {len(np.unique(y))}")

if __name__ == "__main__":
# Examples (uncomment what you need)
    export_amazon_min("Computers", root="./pyg_datasets", out_dir="./amazon_min")
    export_amazon_min("Photo", root="./pyg_datasets", out_dir="./amazon_min")

Processing...
Done!


✓ Wrote edges.txt, features.csv, labels.csv to: ./amazon_min\computers
 Nodes: 13,752 | Feats: 767 | Edges: 491,722 | Classes: 10


Processing...
Done!


✓ Wrote edges.txt, features.csv, labels.csv to: ./amazon_min\photo
 Nodes: 7,650 | Feats: 745 | Edges: 238,162 | Classes: 8
